<a href="https://colab.research.google.com/github/ash1shm/AG-PortfolioDoctor/blob/main/research_assistant_eval_workshop.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Building & Evaluating a Research Assistant Agent with Arize AX

**Hands-on notebook** — adapted from the [Arize AX workshop](https://youtu.be/qZj7sqmidqA) by Laurie Voss.

Over ~90 minutes we'll build a research assistant agent and put it through the full evaluation lifecycle:

1. **Set up tracing** — instrument a Claude agent with Arize AX
2. **Build the agent** — a two-turn research assistant (search → synthesize)
3. **Generate test data** — diverse research queries
4. **Look at your data** — error analysis before writing any evals
5. **Write a code eval** — deterministic source-citation check
6. **Run built-in evals** — correctness vs. faithfulness
7. **Write a custom rubric** — a thoroughness eval from scratch
8. **Meta-evaluate** — test the judge against human labels
9. **Improve the agent** — feed explanations back to Claude, run an experiment

### You'll need:
- An **Anthropic API key** → [console.anthropic.com](https://console.anthropic.com)
- An **Arize AX account** → [arize.com](https://arize.com) (free trial)
- Your **Arize Space ID** (in workspace settings)
- An **Arize API key** (generate in Settings)

> Run in Colab: **File → Save a copy in Drive** so you can edit your own version.

## Step 1: Set Up Tracing

Tracing captures a structured record of every step your agent takes — every LLM call, tool invocation, and decision — with inputs and outputs at each point. Set it up once and data flows automatically.

In [ ]:
%pip install -q claude-agent-sdk openinference-instrumentation-claude-agent-sdk arize arize-otel arize-phoenix anthropic

In [ ]:
import os
from google.colab import userdata

anthropic_api_key = userdata.get('anthropic-api-key')
arize_api_key = userdata.get('arize-api-key')
arize_space_id = userdata.get('arize-space-id')

os.environ["ANTHROPIC_API_KEY"] = anthropic_api_key
os.environ["ARIZE_API_KEY"] = arize_api_key
os.environ["ARIZE_SPACE_ID"] = arize_space_id

In [ ]:
from arize.otel import register, Endpoint
from openinference.instrumentation.claude_agent_sdk import ClaudeAgentSDKInstrumentor

tracer_provider = register(
    space_id=os.environ["ARIZE_SPACE_ID"],
    api_key=os.environ["ARIZE_API_KEY"],
    project_name="research-assistant-demo",
    endpoint=Endpoint.ARIZE,
    batch=False,
)

# Auto-instrument the Claude Agent SDK — every LLM and tool call becomes a span.
ClaudeAgentSDKInstrumentor().instrument(tracer_provider=tracer_provider)

In [ ]:
from arize import ArizeClient

arize_client = ArizeClient(api_key=os.environ["ARIZE_API_KEY"])
SPACE_ID = os.environ["ARIZE_SPACE_ID"]
PROJECT_NAME = "research-assistant-demo"

## Step 2: Build the Agent

We're building a research assistant using the Claude Agent SDK. The agent works in two turns:

- **Turn 1: Research** — searches the web for information on the topic
- **Turn 2: Synthesize** — compiles findings into a structured, sourced briefing

The `ClaudeSDKClient` maintains conversation context between turns, so the synthesizer has full access to the researcher's findings.

In [ ]:
from claude_agent_sdk import ClaudeSDKClient, ClaudeAgentOptions, AssistantMessage, TextBlock, ResultMessage
from opentelemetry import trace

tracer = trace.get_tracer(__name__)

RESEARCH_PROMPT = """Research the following topic: {topic}

Use web search to find current, authoritative information. Look for:
- Key facts and recent developments
- Multiple perspectives or competing viewpoints
- Specific data points, statistics, or examples
- Expert opinions or official statements"""

SYNTHESIZE_PROMPT = """Based on your research above, write a concise briefing that:
- Summarizes the key findings
- Cites specific sources for major claims
- Identifies areas of consensus and disagreement
- Notes any gaps or limitations in the available information"""

options = ClaudeAgentOptions(
    model="claude-haiku-4-5-20251001",
    allowed_tools=["WebSearch"],
    permission_mode="acceptEdits",
)


async def research_topic(topic: str, verbose: bool = True) -> str:
    with tracer.start_as_current_span(
        "research_topic",
        attributes={"input.value": topic},
    ) as span:
        async with ClaudeSDKClient(options=options) as client:
            # Turn 1: Research
            if verbose:
                print(f"--- Researching: {topic} ---")
            await client.query(RESEARCH_PROMPT.format(topic=topic))
            async for message in client.receive_response():
                if verbose and isinstance(message, AssistantMessage):
                    for block in message.content:
                        if isinstance(block, TextBlock):
                            preview = block.text[:200].replace("\n", " ")
                            print(f"  [research] {preview}...")

            # Turn 2: Synthesize
            if verbose:
                print(f"--- Synthesizing ---")
            await client.query(SYNTHESIZE_PROMPT)
            report = ""
            async for message in client.receive_response():
                if isinstance(message, AssistantMessage):
                    for block in message.content:
                        if isinstance(block, TextBlock):
                            report += block.text
                            if verbose:
                                preview = block.text[:200].replace("\n", " ")
                                print(f"  [writing] {preview}...")
                elif isinstance(message, ResultMessage):
                    report = message.result or report

            span.set_attribute("output.value", report)
    tracer_provider.force_flush()
    return report

### Test the agent with a single query

This will take a minute or two.

In [ ]:
result = await research_topic("What is the current state of nuclear fusion energy research?")
print(result)

## Step 3: Generate Test Data

To run meaningful evals, we need more than one trace. Here are 12 test queries covering different topics, complexity levels, and question types.

In [ ]:
test_queries = [
    "What are the latest breakthroughs in CRISPR gene editing technology?",
    "How is the EU AI Act being implemented and what are its key provisions?",
    "What is the current state of quantum computing commercialization?",
    "Compare the environmental impact of electric vehicles vs hydrogen fuel cells",
    "What are the main arguments for and against universal basic income?",
    "How has remote work affected commercial real estate markets since 2023?",
    "What is the current scientific consensus on microplastics and human health?",
    "What are the leading approaches to carbon capture and storage?",
    "How is AI being used in drug discovery and what results has it produced?",
    "What is the state of global semiconductor manufacturing capacity?",
    "Compare the space programs of NASA, SpaceX, and China's CNSA",
    "What are the cybersecurity implications of post-quantum cryptography?",
]

In [ ]:
for i, topic in enumerate(test_queries):
    print(f"[{i+1}/{len(test_queries)}] {topic[:60]}...")
    await research_topic(topic, verbose=False)
    print(f"  done")

## Step 4: Look at Your Data — Error Analysis

Before writing a single eval, **read your traces**. This is the step most tutorials skip — and it's the most important one. You need to understand what's actually going wrong before you can measure it.

We'll:
1. **Examine traces** — read the input and output of a few runs
2. **Categorize failures** — label each trace with a root cause
3. **Build a frequency table** — see where to focus first

In [ ]:
from datetime import datetime, timedelta

spans_df = arize_client.spans.export_to_df(
    space_id=SPACE_ID,
    project_name=PROJECT_NAME,
    start_time=datetime.utcnow() - timedelta(days=7),
    end_time=datetime.utcnow(),
)

# Filter to top-level agent spans
parent_spans = spans_df[spans_df["parent_id"].isna()].copy()

if "input" not in parent_spans.columns:
    parent_spans.rename(columns={"attributes.input.value": "input"}, inplace=True)
if "output" not in parent_spans.columns:
    parent_spans.rename(columns={"attributes.output.value": "output"}, inplace=True)
parent_spans = parent_spans.loc[:, ~parent_spans.columns.duplicated()]

print(f"Found {len(parent_spans)} top-level spans")

### Examine a few traces

Read the input and a snippet of the output side by side. Look for: unsourced claims, missing perspectives, vague statements, outdated information, hallucinated sources.

In [ ]:
for i, (idx, row) in enumerate(parent_spans.head(4).iterrows()):
    print(f"{'='*60}")
    print(f"TRACE {i+1}")
    print(f"  INPUT:  {row['input']}")
    print(f"  OUTPUT: {str(row['output'])[:400]}...")
    print()

### Categorize failures by root cause

Label each trace with what you observe. Replace the categories below with your own after reading the traces above.

In [ ]:
import pandas as pd

# TODO: Replace these with your own observations after reading traces!
trace_categories = {
    "nuclear fusion": "looks good",
    "CRISPR gene editing": "missing sources",
    "EU AI Act": "looks good",
    "quantum computing": "vague claims",
    "EVs vs hydrogen": "one-sided comparison",
    "universal basic income": "looks good",
    "remote work real estate": "outdated data",
    "microplastics health": "missing sources",
    "carbon capture": "looks good",
    "AI drug discovery": "vague claims",
    "semiconductor manufacturing": "looks good",
    "space programs comparison": "one-sided comparison",
}

categories_df = pd.DataFrame(
    list(trace_categories.items()),
    columns=["trace", "category"]
)
display(categories_df)

### Frequency table — where should we focus?

Count the categories. Frequency × severity = priority.

In [ ]:
print("Root cause frequency:")
print("-" * 35)
counts = categories_df["category"].value_counts()
for category, count in counts.items():
    bar = "\u2588" * count
    print(f"  {category:<25} {count}  {bar}")
print(f"\nTotal traces: {len(categories_df)}")
print(f"Failure rate: {(len(categories_df) - counts.get('looks good', 0)) / len(categories_df):.0%}")

## Step 5: Code Eval — Source Citation Check

Code evals are deterministic — no LLM, no API call, no cost. Our simplest useful check: does the output actually cite sources? A research briefing without sources is just an opinion.

This checks for common citation patterns: URLs, "according to" attributions, and named source references.

In [ ]:
import re
from phoenix.evals import create_evaluator
from openinference.instrumentation import suppress_tracing
from phoenix.evals import evaluate_dataframe


@create_evaluator(name="cites_sources", kind="code")
def cites_sources(input, output):
    """Code eval: does the output cite any sources?"""
    if not output:
        return {"label": "fail", "score": 0, "explanation": "No output"}

    # Pattern 1: URLs
    urls = re.findall(r'https?://[^\s)\]]+', output)

    # Pattern 2: "according to [Source]" attributions
    attributions = re.findall(
        r'(?:according to|reported by|published by|per|cited by)\s+[A-Z][\w\s]+',
        output, re.IGNORECASE
    )

    # Pattern 3: Named sources (e.g., "Nature", "Reuters", "MIT")
    known_sources = [
        "Reuters", "Bloomberg", "Nature", "Science", "MIT", "Stanford",
        "Harvard", "WHO", "NASA", "IEEE", "Lancet", "NEJM", "Gartner",
        "McKinsey", "Pew Research", "World Bank", "IMF", "IEA",
        "Congressional Research Service", "European Commission",
    ]
    named = [s for s in known_sources if s.lower() in output.lower()]

    total_citations = len(urls) + len(attributions) + len(named)

    if total_citations >= 3:
        return {"label": "pass", "score": 1, "explanation": f"Found {total_citations} citations"}
    elif total_citations >= 1:
        return {
            "label": "weak",
            "score": 0.5,
            "explanation": f"Only {total_citations} citation(s) — research briefings should cite 3+",
        }
    else:
        return {"label": "fail", "score": 0, "explanation": "No sources cited"}


print(f"Using {len(parent_spans)} top-level spans for code evals")

with suppress_tracing():
    citation_results = evaluate_dataframe(
        dataframe=parent_spans, evaluators=[cites_sources]
    )

In [ ]:
import pandas as pd

citation_scores = pd.json_normalize(citation_results["cites_sources_score"])
print("Source Citation Results:")
print(f"  {citation_scores['label'].value_counts().to_dict()}")

weak_or_fail = citation_scores[citation_scores["label"].isin(["fail", "weak"])]
if len(weak_or_fail) > 0:
    print(f"\nWeak or missing citations:")
    for _, row in weak_or_fail.iterrows():
        print(f"  {row.get('explanation', 'no explanation')}")
else:
    print("\nAll passed!")

### Send results to Arize AX

In [ ]:
from phoenix.evals.utils import to_annotation_dataframe

def log_eval_to_ax(eval_results_df, eval_name):
    """Convert evaluate_dataframe results to AX format and log them."""
    annotations = to_annotation_dataframe(dataframe=eval_results_df)
    annotations = annotations.rename(columns={
        "label": f"eval.{eval_name}.label",
        "score": f"eval.{eval_name}.score",
        "explanation": f"eval.{eval_name}.explanation",
    })
    for col in ("name", "metadata", "annotation_name", "annotator_kind"):
        if col in annotations.columns:
            annotations = annotations.drop(columns=[col])
    expl_col = f"eval.{eval_name}.explanation"
    if expl_col in annotations.columns:
        annotations[expl_col] = annotations[expl_col].fillna("").astype(str)
    if annotations.index.name == "context.span_id":
        annotations = annotations.reset_index()
    arize_client.spans.update_evaluations(
        space_id=SPACE_ID,
        project_name=PROJECT_NAME,
        dataframe=annotations,
    )
    print(f"Logged {len(annotations)} {eval_name} evaluations to AX")


log_eval_to_ax(citation_results, eval_name="cites_sources")

## Step 6: Built-In Evals — Correctness vs. Faithfulness

Arize AX ships with pre-built evaluators. We'll run two to see the difference:

- **Correctness** — Is the output factually accurate? (judges against the LLM's own knowledge)
- **Faithfulness** — Does the output stick to the sources the agent actually found?

As the workshop showed, correctness often rejects everything when the agent uses live web data. Faithfulness asks the right question: is the report grounded in its research?

In [ ]:
from phoenix.evals.llm import LLM
from phoenix.evals.metrics import CorrectnessEvaluator

# Sonnet judges Haiku's output — larger judge model = more reliable grading.
llm = LLM(provider="anthropic", model="claude-sonnet-4-6")

correctness_eval = CorrectnessEvaluator(llm=llm)

with suppress_tracing():
    correctness_results = evaluate_dataframe(
        dataframe=parent_spans, evaluators=[correctness_eval]
    )

log_eval_to_ax(correctness_results, eval_name="correctness")

### Faithfulness — with the research context

The key difference: we give the judge the agent's research as context, so it checks grounding rather than fact-checking against training data.

In [ ]:
# Extract research context from child spans (Turn 1 output)
child_spans = spans_df[spans_df["parent_id"].notna()].copy()

research_context = {}
for trace_id, group in child_spans.groupby("context.trace_id"):
    llm_spans = group[group["attributes.llm.output_messages"].notna()]
    if len(llm_spans) > 0:
        first_response = llm_spans.iloc[0]
        output = first_response.get("attributes.output.value", "")
        if output:
            research_context[trace_id] = str(output)[:5000]

parent_spans["context"] = parent_spans["context.trace_id"].map(research_context)
print(f"Added context to {parent_spans['context'].notna().sum()}/{len(parent_spans)} spans")

In [ ]:
from phoenix.evals.metrics import FaithfulnessEvaluator

faithfulness_eval = FaithfulnessEvaluator(llm=llm)
spans_with_context = parent_spans[parent_spans["context"].notna()].copy()

with suppress_tracing():
    faith_results = evaluate_dataframe(
        dataframe=spans_with_context, evaluators=[faithfulness_eval]
    )

faith_scores = pd.json_normalize(faith_results["faithfulness_score"])
print("Faithfulness results:")
print(f"  {faith_scores['label'].value_counts().to_dict()}")
print(f"\nCorrectness judges against training data — often useless for live research.")
print(f"Faithfulness judges against the agent's own sources — much more useful.")

log_eval_to_ax(faith_results, eval_name="faithfulness")

## Step 7: Custom Eval — Thoroughness Rubric

Built-in evals are general-purpose. Custom rubrics check what matters for **your** application. Our research assistant should produce thorough, balanced briefings — not just surface-level summaries.

A good eval prompt has four parts (per the AX docs):
1. **Define the judge's role** — domain context
2. **Explicit criteria** — measurable pass/fail conditions
3. **Label data with XML tags** — separate instructions from data
4. **Leave output format out** — labels defined externally via `choices`

In [ ]:
thoroughness_template = """
You are an expert research analyst evaluator. Your task is to judge whether
a research briefing provides thorough, balanced coverage of the topic — not
just a surface-level summary.

THOROUGH — The briefing:
- Covers multiple dimensions of the topic (technical, economic, social, etc.)
- Includes specific data points, statistics, or quantitative evidence
- Presents multiple perspectives or competing viewpoints where they exist
- Identifies nuances, caveats, or areas of ongoing debate
- Distinguishes between established facts and emerging/uncertain claims

NOT THOROUGH — The briefing:
- Only covers one angle or dimension of the topic
- Makes broad claims without supporting evidence
- Presents only one perspective on a contested issue
- Treats uncertain or debated claims as settled fact
- Reads like a Wikipedia intro rather than an analyst's briefing

Example — THOROUGH:
"CRISPR base editing achieved 90% efficiency in sickle cell trials
(Vertex/CRISPR Therapeutics, NEJM 2024), but delivery remains a bottleneck —
lipid nanoparticles reach the liver effectively but not muscle or brain tissue.
The EU approved Casgevy in Feb 2024 at ~$2.2M per treatment, raising access
concerns. Meanwhile, prime editing (Broad Institute) offers more precise edits
but at lower efficiency (~30-50%), creating a tradeoff researchers are actively
working to resolve."

Example — NOT THOROUGH:
"CRISPR is a revolutionary gene editing technology that has shown promise in
treating genetic diseases. Scientists are making progress and several clinical
trials are underway. The technology has both supporters and critics."

<query>
{input}
</query>

<research_briefing>
{output}
</research_briefing>
"""


In [ ]:
from phoenix.evals import ClassificationEvaluator

thoroughness_evaluator = ClassificationEvaluator(
    name="thoroughness",
    llm=llm,
    prompt_template=thoroughness_template,
    choices={"thorough": 1.0, "not thorough": 0.0},
)

with suppress_tracing():
    thoroughness_results = evaluate_dataframe(
        dataframe=parent_spans, evaluators=[thoroughness_evaluator]
    )

scores = pd.json_normalize(thoroughness_results["thoroughness_score"])
print("Thoroughness results:")
print(f"  {scores['label'].value_counts().to_dict()}")

log_eval_to_ax(thoroughness_results, eval_name="thoroughness")

## Step 8: Meta-Evaluation — Can You Trust Your Judge?

Your LLM judge is a classifier. You can check its predictions against ground truth — your own human judgment. This is **meta-evaluation**: testing the test.

### Label examples in the AX UI

1. Open any trace from `research-assistant-demo`
2. Click **Add Annotation → + New Annotation**
3. Name it `human_thoroughness`, type **Categorical**, values: `thorough` and `not thorough`
4. Label 6+ spans by reading the output and applying your judgment
5. Run the cell below to pull the annotations back

In [ ]:
# Re-pull spans with annotations
spans_df = arize_client.spans.export_to_df(
    space_id=SPACE_ID,
    project_name=PROJECT_NAME,
    start_time=datetime.utcnow() - timedelta(days=7),
    end_time=datetime.utcnow(),
)

parent_spans = spans_df[spans_df["parent_id"].isna()].copy()
if "input" not in parent_spans.columns:
    parent_spans.rename(columns={"attributes.input.value": "input"}, inplace=True)
if "output" not in parent_spans.columns:
    parent_spans.rename(columns={"attributes.output.value": "output"}, inplace=True)
parent_spans = parent_spans.loc[:, ~parent_spans.columns.duplicated()]

ANNOTATION_COL = "annotation.human_thoroughness.label"
if ANNOTATION_COL not in parent_spans.columns:
    raise RuntimeError(
        f"No '{ANNOTATION_COL}' column found. "
        "Have you applied human_thoroughness annotations in the AX UI yet?"
    )

labeled_subset = parent_spans[parent_spans[ANNOTATION_COL].notna()].copy()
print(f"Found {len(labeled_subset)} labeled spans.")
display(labeled_subset[["input", ANNOTATION_COL]])

### Run the judge on the same examples and compare

In [ ]:
with suppress_tracing():
    judge_results = evaluate_dataframe(
        dataframe=labeled_subset, evaluators=[thoroughness_evaluator]
    )

judge_labels = pd.json_normalize(judge_results["thoroughness_score"])
labeled_subset["judge_label"] = judge_labels["label"].values
labeled_subset["judge_explanation"] = judge_labels["explanation"].values

comparison = labeled_subset[["input", ANNOTATION_COL, "judge_label"]].copy()
comparison = comparison.rename(columns={ANNOTATION_COL: "human_label"})
comparison["human_label"] = comparison["human_label"].str.replace("_", " ")
comparison["agree"] = comparison["human_label"] == comparison["judge_label"]

print("Human vs. Judge Comparison")
print("=" * 70)
for i, (idx, row) in enumerate(comparison.iterrows()):
    agree_mark = "\u2713" if row["agree"] else "\u2717 DISAGREE"
    print(f"  {row['input'][:50]:<50}  human: {row['human_label']:<15} judge: {row['judge_label']:<15} {agree_mark}")

agreement_rate = comparison["agree"].mean()
print(f"\nAgreement rate: {agreement_rate:.0%} ({comparison['agree'].sum()}/{len(comparison)})")

# Show disagreements
disagreements = labeled_subset[comparison["human_label"].values != labeled_subset["judge_label"].values]
if len(disagreements) > 0:
    print(f"\n{'='*70}")
    print("DISAGREEMENTS — read the judge's reasoning:")
    for idx, row in disagreements.iterrows():
        human = str(row[ANNOTATION_COL]).replace("_", " ")
        print(f"\n  Query: {row['input'][:60]}")
        print(f"  Human: {human}  |  Judge: {row['judge_label']}")
        print(f"  Reasoning: {row['judge_explanation'][:300]}...")
else:
    print("\nNo disagreements!")

### Precision and recall

In [ ]:
human_fail = set(comparison[comparison["human_label"] == "not thorough"].index)
judge_fail = set(comparison[comparison["judge_label"] == "not thorough"].index)

true_positives = len(human_fail & judge_fail)
false_positives = len(judge_fail - human_fail)
false_negatives = len(human_fail - judge_fail)
true_negatives = len(comparison) - true_positives - false_positives - false_negatives

print("Confusion Matrix (for 'not thorough' class)")
print("=" * 45)
print(f"                    Judge: fail  Judge: pass")
print(f"  Human: fail           {true_positives}            {false_negatives}")
print(f"  Human: pass           {false_positives}            {true_negatives}")
print()

precision = true_positives / (true_positives + false_positives) if (true_positives + false_positives) > 0 else 0
recall = true_positives / (true_positives + false_negatives) if (true_positives + false_negatives) > 0 else 0

print(f"Precision: {precision:.0%}  (when the judge says 'not thorough', is it right?)")
print(f"Recall:    {recall:.0%}  (of all real fails, how many does it catch?)")
print(f"\nWith only {len(comparison)} examples, these numbers are rough.")
print(f"Aim for 20-50 labeled examples for a proper validation.")

## Step 9: Improve the Agent — Automatically

The thoroughness eval told us what's wrong, and its explanations told us **why**. Instead of hand-editing prompts, we feed those explanations to Claude and let it rewrite the agent's prompts — then run an experiment to prove the new prompts are better.

**Before running the experiment cell**, save a failure dataset in AX:
1. Filter traces to `thoroughness == "not thorough"`
2. Click **Save as Dataset**
3. Name it `research-assistant-fails`

In [ ]:
import anthropic

# Collect judge explanations from failing traces
scores = pd.json_normalize(thoroughness_results["thoroughness_score"])
parent_spans["thoroughness_label"] = scores["label"].values
parent_spans["thoroughness_explanation"] = scores["explanation"].values

failing = parent_spans[parent_spans["thoroughness_label"] == "not thorough"].copy()
print(f"{len(failing)} of {len(parent_spans)} traces failed the thoroughness eval\n")

REQUIREMENTS = """The research assistant must produce briefings that:
- Cover multiple dimensions of the topic (technical, economic, social, etc.)
- Include specific data points, statistics, and quantitative evidence
- Present multiple perspectives or competing viewpoints
- Cite specific sources for major claims
- Distinguish established facts from emerging or uncertain claims
- For comparison queries, include balanced coverage of all items compared"""

explanations = "\n".join(
    f"- {row['thoroughness_explanation']}" for _, row in failing.iterrows()
)

improvement_prompt = f"""You are improving the prompts of a research assistant agent.
The agent runs in two turns: a RESEARCH turn and a SYNTHESIZE turn.

An LLM judge scored the agent's briefings for "thoroughness" and explained
every failure. Find the RECURRING THEMES across these explanations and
rewrite the two prompts to address them.

## CURRENT RESEARCH PROMPT
{RESEARCH_PROMPT}

## CURRENT SYNTHESIZE PROMPT
{SYNTHESIZE_PROMPT}

## REQUIREMENTS
{REQUIREMENTS}

## JUDGE EXPLANATIONS FOR FAILING BRIEFINGS
{explanations}

Return the two revised prompts in XML tags:
<research_prompt>...revised research prompt...</research_prompt>
<synthesize_prompt>...revised synthesize prompt...</synthesize_prompt>
Keep the {{topic}} placeholder in the research prompt."""

print(improvement_prompt[:500] + "\n...")

In [ ]:
import re as re_module

anthropic_client = anthropic.Anthropic()

with suppress_tracing():
    response = anthropic_client.messages.create(
        model="claude-sonnet-4-6",
        max_tokens=2000,
        messages=[{"role": "user", "content": improvement_prompt}],
    )
reply = response.content[0].text

IMPROVED_RESEARCH_PROMPT = re_module.search(
    r"<research_prompt>(.*?)</research_prompt>", reply, re_module.DOTALL
).group(1).strip()
IMPROVED_SYNTHESIZE_PROMPT = re_module.search(
    r"<synthesize_prompt>(.*?)</synthesize_prompt>", reply, re_module.DOTALL
).group(1).strip()

print("=== IMPROVED RESEARCH PROMPT ===")
print(IMPROVED_RESEARCH_PROMPT)
print("\n=== IMPROVED SYNTHESIZE PROMPT ===")
print(IMPROVED_SYNTHESIZE_PROMPT)

### Wire up the improved agent and run the experiment

In [ ]:
async def improved_research_topic(topic: str) -> str:
    with tracer.start_as_current_span(
        "research_topic",
        attributes={"input.value": topic},
    ) as span:
        async with ClaudeSDKClient(options=options) as client:
            research = IMPROVED_RESEARCH_PROMPT.replace("{topic}", topic)
            await client.query(research)
            async for message in client.receive_response():
                pass

            await client.query(IMPROVED_SYNTHESIZE_PROMPT)
            report = ""
            async for message in client.receive_response():
                if isinstance(message, AssistantMessage):
                    for block in message.content:
                        if isinstance(block, TextBlock):
                            report += block.text
                elif isinstance(message, ResultMessage):
                    report = message.result or report

            span.set_attribute("output.value", report)
    tracer_provider.force_flush()
    return report

In [ ]:
import asyncio
import concurrent.futures

try:
    from arize.experiments import EvaluationResult
except ImportError:
    from arize.experimental.datasets.experiments.evaluators.base import EvaluationResult


def run_async(coro):
    with concurrent.futures.ThreadPoolExecutor(1) as pool:
        return pool.submit(lambda: asyncio.run(coro)).result()


def improved_agent_task(dataset_row):
    topic = dataset_row.get("attributes.input.value", "")
    return run_async(improved_research_topic(topic))


def thoroughness_eval_fn(output, dataset_row):
    inp = dataset_row.get("attributes.input.value", "")
    scores = thoroughness_evaluator.evaluate({"input": inp, "output": output})
    s = scores[0]
    return EvaluationResult(
        score=float(s.score) if s.score is not None else 0.0,
        label=s.label,
        explanation=getattr(s, "explanation", "") or "",
    )


experiment, experiment_df = arize_client.experiments.run(
    name="improved-prompts-v1",
    dataset="research-assistant-fails",
    space=SPACE_ID,
    task=improved_agent_task,
    evaluators=[thoroughness_eval_fn],
    concurrency=2,
)

print(f"Experiment complete: {len(experiment_df)} examples scored.")
experiment_df

## What's Next?

You've run the full loop: **trace → eval → explanation → improved prompt → experiment**.

To take this to production:

- **Online evals** — run your thoroughness rubric continuously on new traffic in the AX UI
- **Monitors** — set alerts when thoroughness or citation rates drop
- **Coding agent fixes** — use the [arize-skills](https://github.com/Arize-ai/arize-skills) plugin to let a coding agent investigate failures and propose fixes as PRs

The central practice: **read actual traces and define success before automating the grading.**